# Can we predict `passed_quiz`?
Protocol: `protocol.yaml`. Group holdout on `student_key` (rows in `holdout.csv`), 5-fold GroupKFold CV on training rows, choose by mean CV ROC AUC, refit once and score on holdout.

**Feature timing.** The quiz closes the session, so predictions are made before the quiz. `session_minutes` is the length of the whole session, which includes the quiz, so it is only known after the outcome and is excluded. `student_key` is excluded per protocol. Used: `enrol_year`, `school_code`, `prior_score`, `hints_used` (hints are opened during the session, before the quiz).

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
df=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
FEATURES=['enrol_year','school_code','prior_score','hints_used']
TARGET='passed_quiz'
tr,ho=df.iloc[train_idx],df.iloc[hold_idx]
assert not set(tr.student_key)&set(ho.student_key)
print(len(tr),len(ho),tr[TARGET].mean(),ho[TARGET].mean())

## Candidates (named before any scoring)

In [ ]:
candidates={
 'baseline_prior':lambda: DummyClassifier(strategy='prior'),
 'logreg_C1':lambda: make_pipeline(StandardScaler(),LogisticRegression(C=1.0,max_iter=1000)),
 'logreg_C0.1':lambda: make_pipeline(StandardScaler(),LogisticRegression(C=0.1,max_iter=1000)),
 'hgb_depth3':lambda: HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0),
 'hgb_depth5':lambda: HistGradientBoostingClassifier(max_depth=5,learning_rate=0.05,max_iter=200,random_state=0),
}
list(candidates)

## Cross-validation on training rows (preprocessing is inside each pipeline, fit per fold)

In [ ]:
gkf=GroupKFold(n_splits=5)
X,y,g=tr[FEATURES].to_numpy(),tr[TARGET].to_numpy(),tr['student_key'].to_numpy()
folds=list(gkf.split(X,y,g))
cv_scores={}
for name,mk in candidates.items():
    s=[]
    for a,b in folds:
        m=mk().fit(X[a],y[a]); s.append(roc_auc_score(y[b],m.predict_proba(X[b])[:,1]))
    cv_scores[name]=s
cvdf=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
cvdf

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
print('chosen:',chosen)

## Refit on all training rows, score once on holdout

In [ ]:
final=candidates[chosen]().fit(X,y)
holdout_auc=roc_auc_score(ho[TARGET],final.predict_proba(ho[FEATURES].to_numpy())[:,1])
print('holdout roc_auc',holdout_auc)

In [ ]:
evidence={
 'features':FEATURES,
 'candidates':list(candidates),
 'cv_folds':[[train_idx[a].tolist(),train_idx[b].tolist()] for a,b in folds],
 'cv_scores':{k:[float(x) for x in v] for k,v in cv_scores.items()},
 'chosen':chosen,
 'holdout_rows':hold_idx.tolist(),
 'holdout_scores':{chosen:float(holdout_auc)},
}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above; the baseline sits at 0.5, so any gap is the signal available from pre-quiz features. `session_minutes` correlates most strongly with the outcome but is excluded because it is not known before the quiz ends.